# Emotion Recognition - EfficientNet-B0 Training (Colab + GPU)

Bu notebook, GPU uzerinde **EfficientNet-B0 (Transfer Learning)** modelini FERPlus uzerinde egitir ve test sonuclarini gosterir.

**Model:** EfficientNet-B0 (ImageNet pretrained, fine-tuned)

**Dataset:** FERPlus (crowdsourced annotations, 7 sinif)

**Transfer Learning Stratejisi:**
- ImageNet agirliklarindan baslama
- Backbone'un son 2 blogu acik (fine-tuning)
- Yeni classifier head (Dropout -> FC -> ReLU -> Dropout -> FC)
- Dusuk learning rate (1e-4)

**Workflow:** GitHub Clone -> Kaggle Dataset -> GPU Training -> Test Sonuclari

**Onemli:** Runtime > Change runtime type > T4 GPU sectiginizden emin olun!

## 1. GPU Kontrolu

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] GPU bulunamadi! Runtime > Change runtime type > T4 GPU secin.")

## 2. GitHub Repo Klonlama

In [ ]:
import os

REPO_URL = "https://github.com/aysenurhepguven0/Emotion-Aware-Adaptive-Virtual-Interaction-System.git"
REPO_NAME = "Emotion-Aware-Adaptive-Virtual-Interaction-System"

%cd /content
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print(f"{REPO_NAME} zaten mevcut, pull yapiliyor...")
    !cd {REPO_NAME} && git pull

%cd /content/{REPO_NAME}
print(f"[OK] Calisma dizini: {os.getcwd()}")

## 3. Bagimliliklari Yukleme

In [ ]:
!pip install -q hsemotion --no-deps
!pip install -q kagglehub

## 4. Dataset Indirme (KaggleHub)

KaggleHub ile FER2013 ve FERPlus datasetlerini indirir.

Kaggle hesabinizla otomatik authenticate olur.

In [ ]:
import kagglehub
import shutil
import os

# ----- FER2013 -----
print("FER2013 indiriliyor...")
fer_path = kagglehub.dataset_download("msambare/fer2013")
print(f"FER2013 path: {fer_path}")

# data/fer2013 klasorune kopyala
os.makedirs("data/fer2013", exist_ok=True)
!cp -r {fer_path}/* data/fer2013/
print("[OK] FER2013 yuklendi")
!ls data/fer2013/

In [ ]:
# ----- FER+ (FERPlus) -----
print("FERPlus indiriliyor...")
ferplus_path = kagglehub.dataset_download("arnabkumarroy02/ferplus")
print(f"FERPlus path: {ferplus_path}")

# data/ferplus klasorune kopyala
os.makedirs("data/ferplus", exist_ok=True)
!cp -r {ferplus_path}/* data/ferplus/
print("[OK] FERPlus yuklendi")
!ls data/ferplus/

In [ ]:
# Dataset yapisini kontrol et
import os

for ds_name in ['fer2013', 'ferplus']:
    ds_path = f'data/{ds_name}'
    if os.path.exists(ds_path):
        print(f"\n{ds_name}/")
        for split in sorted(os.listdir(ds_path)):
            split_path = os.path.join(ds_path, split)
            if os.path.isdir(split_path):
                classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
                total = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
                print(f"  {split}: {total:,} images ({len(classes)} classes)")
    else:
        print(f"[ERROR] {ds_name} bulunamadi!")

## 5. Config Ayarlari (GPU + EfficientNet)

**EfficientNet-B0 Transfer Learning parametreleri:**
- Input size: 224x224 (ImageNet standard)
- Batch size: 32 (buyuk resim, kucuk batch)
- Learning rate: 1e-4 (transfer learning icin dusuk)
- Backbone: Son 2 blok acik (fine-tuning)

In [ ]:
import torch
import config

# EfficientNet config kontrolu
eff_config = config.MODEL_CONFIGS["efficientnet"]
print(f"EfficientNet Config:")
print(f"  img_size:     {eff_config['img_size']}")
print(f"  num_channels: {eff_config['num_channels']}")
print(f"  batch_size:   {eff_config['batch_size']}")
print(f"  lr:           {eff_config['lr']}")

if torch.cuda.is_available():
    config.EPOCHS = 30
    config.MODEL_CONFIGS["efficientnet"]["batch_size"] = 32
    print(f"\n[GPU] {config.EPOCHS} epoch, batch_size=32")
else:
    config.EPOCHS = 10
    print(f"\n[CPU] {config.EPOCHS} epoch")

print(f"Device: {config.DEVICE}")

## 6. EfficientNet-B0 Model Egitimi

FERPlus uzerinde EfficientNet-B0 (Transfer Learning) egitir.

**Transfer Learning detaylari:**
- ImageNet pretrained agirliklar
- Backbone buyuk kismi frozen (genel ozellikler korunur)
- Son 2 blok + classifier fine-tuned (emotion recognition icin)
- ~5.3M parametre (4M frozen + 1.3M trainable)

In [ ]:
# ============================================
# DATASET SECIN
# ============================================
DATASET = "ferplus"  # "fer2013", "ferplus", "rafdb", "ckplus"

!python main.py --mode train --model efficientnet --dataset {DATASET}


## 7. Test Sonuclari (EfficientNet-B0)

Egitilen modeli test setinde degerlendirir.

Metrikler: Accuracy, Precision, Recall, F1-score, Confusion Matrix

In [ ]:
import os
import config
from evaluate import evaluate_model
from models.efficientnet import get_efficientnet_model

MODEL_PATH = config.BEST_MODEL_PATHS["efficientnet"]
if not os.path.exists(MODEL_PATH):
    print("[ERROR] EfficientNet modeli bulunamadi. Once egitim yapin.")
else:
    print(f"[INFO] Model yukleniyor: {MODEL_PATH}")
    model = get_efficientnet_model(pretrained_path=MODEL_PATH)
    results = evaluate_model(model=model, dataset_name=DATASET, split_name=f"{DATASET} Test")
    y_true = results["y_true"]
    y_pred = results["y_pred"]

    neutral_label = next(
        k for k, v in config.EMOTION_LABELS.items() if v.lower() == "neutral"
    )
    neutral_mask = y_true == neutral_label
    if neutral_mask.any():
        neutral_acc = (y_pred[neutral_mask] == neutral_label).mean() * 100
        print(f"Neutral class accuracy: {neutral_acc:.2f}% (n={neutral_mask.sum()})")
    else:
        print("[WARNING] Neutral sinifi test setinde bulunamadi.")

## 8. Mini-Xception vs EfficientNet-B0 Karsilastirmasi

Onceki Mini-Xception sonuclariyla karsilastirma.

In [ ]:
import json
import os

print("="*60)
print("  MODEL KARSILASTIRMASI (FERPlus)")
print("="*60)

# Mini-Xception sonuclari (onceki egitimden)
mini_history_path = "outputs/training_history_ferplus_mini_xception.json"
eff_history_path = "outputs/training_history_ferplus_efficientnet.json"

results_table = []

if os.path.exists(mini_history_path):
    with open(mini_history_path, 'r') as f:
        mini_history = json.load(f)
    mini_best_acc = max(mini_history.get('val_acc', [0]))
    results_table.append(("Mini-Xception", "347K", mini_best_acc))
else:
    print("[INFO] Mini-Xception gecmisi bulunamadi (onceki notebook'ta egitilmemis olabilir)")
    results_table.append(("Mini-Xception", "347K", 66.68))  # Onceki sonuc

if os.path.exists(eff_history_path):
    with open(eff_history_path, 'r') as f:
        eff_history = json.load(f)
    eff_best_acc = max(eff_history.get('val_acc', [0]))
    results_table.append(("EfficientNet-B0", "~5.3M", eff_best_acc))
else:
    print("[INFO] EfficientNet gecmisi bulunamadi")

print(f"\n{'Model':<20} {'Params':<12} {'Best Val Acc':>12}")
print("-"*46)
for name, params, acc in results_table:
    print(f"{name:<20} {params:<12} {acc:>11.2f}%")
print("="*60)

## 9. Sonuclari Google Drive'a Kaydetme (Opsiyonel)

Egitilen modeli ve plot'lari Drive'a kaydeder.

In [ ]:
# Google Drive baglama (opsiyonel)
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# DRIVE_DIR = "/content/drive/MyDrive/emotion_recognition_results"
# os.makedirs(DRIVE_DIR, exist_ok=True)

# # Model kaydet
# shutil.copy2(config.BEST_MODEL_PATHS["efficientnet"], DRIVE_DIR)
# print(f"[OK] Model Drive'a kaydedildi: {DRIVE_DIR}")

# # Plot'lari kaydet
# for f in os.listdir("outputs/plots"):
#     shutil.copy2(os.path.join("outputs/plots", f), DRIVE_DIR)
# print(f"[OK] Plot'lar Drive'a kaydedildi")